# Add Strength-State Labels

This notebook adds game-state labels to the origin-shot chain dataset.

The primary goal is to identify true 5v5 chains using stint data:

- 5 home skaters
- 5 away skaters
- no home empty net
- no away empty net
- origin shot time falls inside the matched stint interval

This notebook does not perform the final outside-shot analysis. It creates a labeled dataset for the next notebook.

Rush, in-zone, and faceoff context labels are intentionally deferred. The first preliminary analysis will use 5v5 chains only.

## 1. Setup

In [1]:
# Import libraries and configure paths

from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
# Define project paths

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_RAW = PROJECT_ROOT / "data" / "raw" / "halo_2026"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

print("Project root:", PROJECT_ROOT)
print("Raw data exists:", DATA_RAW.exists())
print("Processed data exists:", DATA_PROCESSED.exists())

Project root: c:\Users\rinal\hockey-analytics\outside-shot-value
Raw data exists: True
Processed data exists: True


## 2. Load Inputs

Load the chain-level dataset from Notebook 05 and the raw game-state tables.

Inputs:

- `origin_shot_sequences_with_tracking.parquet`
- `games.parquet`
- `stints.parquet`

The chain table has one row per origin-shot chain. The stints table is player-level and must be collapsed before joining.

In [3]:
# Load chain-level tracking features and raw game-state tables

chains = pd.read_parquet(DATA_PROCESSED / "origin_shot_sequences_with_tracking.parquet")
games = pd.read_parquet(DATA_RAW / "games.parquet")
stints = pd.read_parquet(DATA_RAW / "stints.parquet")

print("chains:", chains.shape)
print("games:", games.shape)
print("stints:", stints.shape)

chains: (48673, 55)
games: (480, 12)
stints: (2212064, 15)


In [4]:
# Check that the required columns are present before building strength labels

required_chain_columns = [
    "chain_id",
    "game_id",
    "period",
    "team_id",
    "anchor_origin_period_time",
    "anchor_origin_location",
    "observed_origin_attackers_in_slot",
    "observed_origin_defenders_in_slot",
    "origin_tracking_available",
    "origin_tracking_error_flag",
]

required_game_columns = [
    "game_id",
    "home_team",
    "away_team",
    "home_team_id",
    "away_team_id",
]

required_stint_columns = [
    "game_id",
    "period",
    "period_time_start",
    "period_time_end",
    "game_stint",
    "n_home_skaters",
    "n_away_skaters",
    "is_home_net_empty",
    "is_away_net_empty",
    "home_score",
    "away_score",
]

missing_columns = {
    "chains": [c for c in required_chain_columns if c not in chains.columns],
    "games": [c for c in required_game_columns if c not in games.columns],
    "stints": [c for c in required_stint_columns if c not in stints.columns],
}

missing_columns

{'chains': [], 'games': [], 'stints': []}

## 3. Prepare Stint Intervals

The stints table is player-level. Each game-state interval appears once per player on the ice, so joining chains directly to raw stints would multiply rows.

Before joining, reduce stints to one row per unique game/period/stint interval.

In [5]:
# Convert stint start and end times to numeric seconds

stints_clean = stints.copy()

stints_clean["period_time_start"] = pd.to_numeric(
    stints_clean["period_time_start"],
    errors="coerce",
)

stints_clean["period_time_end"] = pd.to_numeric(
    stints_clean["period_time_end"],
    errors="coerce",
)

stints_clean[["period_time_start", "period_time_end"]].describe()

,period_time_start,period_time_end
count,2.212064e+06,2.212064e+06
mean,5.862644e+02,5.955204e+02
std,3.436104e+02,3.437282e+02
min,0.000000e+00,3.000000e-02
25%,2.858700e+02,2.943300e+02
50%,5.820300e+02,5.910300e+02
75%,8.841300e+02,8.930300e+02
max,1.199230e+03,1.201000e+03


In [6]:
# Collapse player-level stint rows into unique game-state intervals

stint_intervals = (
    stints_clean[
        [
            "game_id",
            "period",
            "game_stint",
            "period_time_start",
            "period_time_end",
            "n_home_skaters",
            "n_away_skaters",
            "is_home_net_empty",
            "is_away_net_empty",
            "home_score",
            "away_score",
        ]
    ]
    .drop_duplicates()
    .copy()
)

stint_intervals = stint_intervals.sort_values(
    ["game_id", "period", "period_time_start", "period_time_end", "game_stint"]
).reset_index(drop=True)

print(stint_intervals.shape)
stint_intervals.head()

(187807, 11)


,game_id,period,game_stint,period_time_start,period_time_end,n_home_skaters,n_away_skaters,is_home_net_empty,is_away_net_empty,home_score,away_score
0,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,1,0.0,31.0,5,5,False,False,0,0
1,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,2,31.0,36.0,5,5,False,False,0,0
2,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,3,36.0,38.0,5,5,False,False,0,0
3,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,4,38.0,42.0,5,5,False,False,0,0
4,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,5,42.0,43.0,5,5,False,False,0,0


In [7]:
# Check that each stint interval is uniquely defined and has valid time bounds

stint_interval_validation = {
    "rows": len(stint_intervals),
    "duplicate_game_period_stint": stint_intervals.duplicated(
        ["game_id", "period", "game_stint"]
    ).sum(),
    "missing_start_time": stint_intervals["period_time_start"].isna().sum(),
    "missing_end_time": stint_intervals["period_time_end"].isna().sum(),
    "negative_or_zero_duration": (
        stint_intervals["period_time_end"] <= stint_intervals["period_time_start"]
    ).sum(),
}

stint_interval_validation

{'rows': 187807,
 'duplicate_game_period_stint': np.int64(0),
 'missing_start_time': np.int64(0),
 'missing_end_time': np.int64(0),
 'negative_or_zero_duration': np.int64(0)}

## 4. Attach Team Metadata

The chain table contains the analytical attacking `team_id`, but the stint table reports manpower as home and away skater counts.

To assign the correct team-side strength, attach home and away team IDs from the games table.

In [8]:
# Attach home and away team metadata to each chain

game_team_map = games[
    [
        "game_id",
        "home_team",
        "away_team",
        "home_team_id",
        "away_team_id",
    ]
].copy()

chains_labeled = chains.merge(
    game_team_map,
    how="left",
    on="game_id",
    validate="many_to_one",
)

chains_labeled.shape

(48673, 59)

In [9]:
# Label each chain team as home, away, or unknown

chains_labeled["team_side"] = np.select(
    [
        chains_labeled["team_id"].eq(chains_labeled["home_team_id"]),
        chains_labeled["team_id"].eq(chains_labeled["away_team_id"]),
    ],
    [
        "home",
        "away",
    ],
    default="unknown",
)

team_side_validation = {
    "rows": len(chains_labeled),
    "missing_home_team_id": chains_labeled["home_team_id"].isna().sum(),
    "missing_away_team_id": chains_labeled["away_team_id"].isna().sum(),
    "team_side_counts": chains_labeled["team_side"].value_counts(dropna=False).to_dict(),
}

team_side_validation

{'rows': 48673,
 'missing_home_team_id': np.int64(0),
 'missing_away_team_id': np.int64(0),
 'team_side_counts': {'home': 24836, 'away': 23837}}

## 5. Match Chains To Active Stints

Each valid chain is matched to the stint interval active at the origin shot time.

`pd.merge_asof()` cannot handle null merge keys, so chains with missing origin time are separated before the join and reattached afterward. These are the known invalid origin-link rows and should remain in the dataset as unmatched, not silently dropped.

The join uses:

- `game_id`
- `period`
- latest `period_time_start` before or equal to `anchor_origin_period_time`

After the join, validate that the origin time falls inside the matched stint interval.

In [10]:
# Prepare chain origin times and stint interval starts for matching

chains_for_stint_join = chains_labeled.copy()

chains_for_stint_join["anchor_origin_period_time"] = pd.to_numeric(
    chains_for_stint_join["anchor_origin_period_time"],
    errors="coerce",
)

chains_for_stint_join["_row_order"] = np.arange(len(chains_for_stint_join))

stint_intervals_for_join = stint_intervals.sort_values(
    ["game_id", "period", "period_time_start", "game_stint"]
).reset_index(drop=True)

chains_for_stint_join["anchor_origin_period_time"].isna().sum()

np.int64(2)

In [11]:
# Split chains with valid origin times from known invalid origin-time rows

chains_with_valid_origin_time = chains_for_stint_join[
    chains_for_stint_join["anchor_origin_period_time"].notna()
].copy()

chains_with_missing_origin_time = chains_for_stint_join[
    chains_for_stint_join["anchor_origin_period_time"].isna()
].copy()

{
    "valid_origin_time_rows": len(chains_with_valid_origin_time),
    "missing_origin_time_rows": len(chains_with_missing_origin_time),
}

{'valid_origin_time_rows': 48671, 'missing_origin_time_rows': 2}

In [12]:
# Match each valid chain to the active stint interval within its game and period

stint_right_columns = [
    "game_stint",
    "period_time_end",
    "n_home_skaters",
    "n_away_skaters",
    "is_home_net_empty",
    "is_away_net_empty",
    "home_score",
    "away_score",
]

matched_groups = []

stint_groups = {
    key: group.sort_values("period_time_start").reset_index(drop=True)
    for key, group in stint_intervals_for_join.groupby(["game_id", "period"], sort=False)
}

for key, chain_group in chains_with_valid_origin_time.groupby(["game_id", "period"], sort=False):
    chain_group = chain_group.sort_values("anchor_origin_period_time").reset_index(drop=True)
    stint_group = stint_groups.get(key)

    if stint_group is None or stint_group.empty:
        matched_group = chain_group.copy()
        matched_group["period_time_start"] = pd.NA
        for col in stint_right_columns:
            matched_group[col] = pd.NA
    else:
        matched_group = pd.merge_asof(
            chain_group,
            stint_group[["period_time_start"] + stint_right_columns].sort_values("period_time_start"),
            left_on="anchor_origin_period_time",
            right_on="period_time_start",
            direction="backward",
            allow_exact_matches=True,
        )

    matched_groups.append(matched_group)

chains_with_matched_stints = pd.concat(matched_groups, ignore_index=True)

chains_with_matched_stints.shape

(48671, 70)

In [13]:
# Reattach chains with missing origin time as unmatched stint rows

chains_with_missing_origin_time = chains_with_missing_origin_time.copy()

chains_with_missing_origin_time["period_time_start"] = pd.NA

for col in stint_right_columns:
    chains_with_missing_origin_time[col] = pd.NA

chains_with_stints = pd.concat(
    [
        chains_with_matched_stints,
        chains_with_missing_origin_time,
    ],
    ignore_index=True,
)

chains_with_stints = (
    chains_with_stints
    .sort_values("_row_order")
    .drop(columns=["_row_order"])
    .reset_index(drop=True)
)

chains_with_stints.shape

(48673, 69)

In [14]:
# Coerce stint-match time columns to numeric and validate interval containment

time_cols = [
    "anchor_origin_period_time",
    "period_time_start",
    "period_time_end",
]

for col in time_cols:
    chains_with_stints[col] = pd.to_numeric(
        chains_with_stints[col],
        errors="coerce",
    )

chains_with_stints["origin_time_within_stint"] = (
    chains_with_stints["anchor_origin_period_time"].notna()
    & chains_with_stints["period_time_start"].notna()
    & chains_with_stints["period_time_end"].notna()
    & (chains_with_stints["anchor_origin_period_time"] >= chains_with_stints["period_time_start"])
    & (chains_with_stints["anchor_origin_period_time"] <= chains_with_stints["period_time_end"])
)

stint_join_validation = {
    "rows": len(chains_with_stints),
    "missing_origin_time": chains_with_stints["anchor_origin_period_time"].isna().sum(),
    "missing_stint_match": chains_with_stints["game_stint"].isna().sum(),
    "origin_time_not_within_stint": (
        chains_with_stints["anchor_origin_period_time"].notna()
        & ~chains_with_stints["origin_time_within_stint"]
    ).sum(),
    "duplicate_chain_ids": chains_with_stints.duplicated("chain_id").sum(),
}

stint_join_validation

{'rows': 48673,
 'missing_origin_time': np.int64(2),
 'missing_stint_match': np.int64(2),
 'origin_time_not_within_stint': np.int64(0),
 'duplicate_chain_ids': np.int64(0)}

In [15]:
# Assert the expected active-stint validation results

expected_stint_join_validation = {
    "rows": 48673,
    "missing_origin_time": 2,
    "missing_stint_match": 2,
    "origin_time_not_within_stint": 0,
    "duplicate_chain_ids": 0,
}

for key, expected_value in expected_stint_join_validation.items():
    actual_value = int(stint_join_validation[key])
    assert actual_value == expected_value, (
        f"{key}: expected {expected_value}, got {actual_value}"
    )

print("Stint join validation passed.")

Stint join validation passed.


In [16]:
# Confirm active-stint containment field exists before creating strength labels

assert "origin_time_within_stint" in chains_with_stints.columns
chains_with_stints["origin_time_within_stint"].value_counts(dropna=False)

origin_time_within_stint
True     48671
False        2
Name: count, dtype: int64

## 6. Create Strength-State Labels

Create true game-state labels from the matched stint.

The primary 5v5 definition is:

- home skaters = 5
- away skaters = 5
- home net not empty
- away net not empty
- origin time falls inside the matched stint interval

This is independent of tracking completeness.

In [17]:
# Create the true 5v5 flag from stint manpower and empty-net state

chains_with_stints["is_5v5"] = (
    chains_with_stints["n_home_skaters"].eq(5)
    & chains_with_stints["n_away_skaters"].eq(5)
    & chains_with_stints["is_home_net_empty"].eq(False)
    & chains_with_stints["is_away_net_empty"].eq(False)
    & chains_with_stints["origin_time_within_stint"].eq(True)
)

chains_with_stints["is_5v5"].value_counts(dropna=False)

is_5v5
True     36357
False    12316
Name: count, dtype: int64

In [18]:
# Create team and opponent skater counts from home-away manpower fields

chains_with_stints["team_n_skaters"] = np.select(
    [
        chains_with_stints["team_side"].eq("home"),
        chains_with_stints["team_side"].eq("away"),
    ],
    [
        chains_with_stints["n_home_skaters"],
        chains_with_stints["n_away_skaters"],
    ],
    default=np.nan,
)

chains_with_stints["opponent_n_skaters"] = np.select(
    [
        chains_with_stints["team_side"].eq("home"),
        chains_with_stints["team_side"].eq("away"),
    ],
    [
        chains_with_stints["n_away_skaters"],
        chains_with_stints["n_home_skaters"],
    ],
    default=np.nan,
)

chains_with_stints[
    [
        "team_side",
        "n_home_skaters",
        "n_away_skaters",
        "team_n_skaters",
        "opponent_n_skaters",
    ]
].head()

,team_side,n_home_skaters,n_away_skaters,team_n_skaters,opponent_n_skaters
0,away,4,4,4,4
1,home,5,5,5,5
2,home,5,5,5,5
3,home,5,5,5,5
4,home,5,5,5,5


In [19]:
# Create team and opponent empty-net flags from home-away empty-net fields

chains_with_stints["team_net_empty"] = np.select(
    [
        chains_with_stints["team_side"].eq("home"),
        chains_with_stints["team_side"].eq("away"),
    ],
    [
        chains_with_stints["is_home_net_empty"],
        chains_with_stints["is_away_net_empty"],
    ],
    default=pd.NA,
)

chains_with_stints["opponent_net_empty"] = np.select(
    [
        chains_with_stints["team_side"].eq("home"),
        chains_with_stints["team_side"].eq("away"),
    ],
    [
        chains_with_stints["is_away_net_empty"],
        chains_with_stints["is_home_net_empty"],
    ],
    default=pd.NA,
)

chains_with_stints[
    [
        "team_side",
        "is_home_net_empty",
        "is_away_net_empty",
        "team_net_empty",
        "opponent_net_empty",
    ]
].head()

,team_side,is_home_net_empty,is_away_net_empty,team_net_empty,opponent_net_empty
0,away,False,False,False,False
1,home,False,False,False,False
2,home,False,False,False,False
3,home,False,False,False,False
4,home,False,False,False,False


In [20]:
# Summarize the derived strength-state labels

strength_summary = {
    "rows": len(chains_with_stints),
    "is_5v5_counts": chains_with_stints["is_5v5"].value_counts(dropna=False).to_dict(),
    "team_skaters_counts": chains_with_stints["team_n_skaters"].value_counts(dropna=False).sort_index().to_dict(),
    "opponent_skaters_counts": chains_with_stints["opponent_n_skaters"].value_counts(dropna=False).sort_index().to_dict(),
    "team_net_empty_counts": chains_with_stints["team_net_empty"].value_counts(dropna=False).to_dict(),
    "opponent_net_empty_counts": chains_with_stints["opponent_net_empty"].value_counts(dropna=False).to_dict(),
}

strength_summary

{'rows': 48673,
 'is_5v5_counts': {True: 36357, False: 12316},
 'team_skaters_counts': {3: 477, 4: 2216, 5: 44824, 6: 1154, <NA>: 2},
 'opponent_skaters_counts': {3: 1083, 4: 9044, 5: 38338, 6: 206, <NA>: 2},
 'team_net_empty_counts': {False: 47484, True: 1187, <NA>: 2},
 'opponent_net_empty_counts': {False: 48456, True: 215, <NA>: 2}}

## 7. Cohort Waterfall

Build the denominator waterfall for the next analysis notebook.

This table separates true 5v5 filtering from tracking availability and tracking error flags.

In [21]:
# Build the denominator waterfall for preliminary 5v5 outside-shot analysis

is_5v5 = chains_with_stints["is_5v5"]
is_outside_origin = chains_with_stints["anchor_origin_location"].eq("outside")
has_tracking_available = chains_with_stints["origin_tracking_available"]
has_no_tracking_error = ~chains_with_stints["origin_tracking_error_flag"]

cohort_waterfall = pd.DataFrame(
    [
        {
            "cohort": "all_origin_shot_chains",
            "chains": len(chains_with_stints),
        },
        {
            "cohort": "valid_origin_time_and_stint",
            "chains": int(chains_with_stints["origin_time_within_stint"].sum()),
        },
        {
            "cohort": "is_5v5",
            "chains": int(is_5v5.sum()),
        },
        {
            "cohort": "is_5v5_outside_origin",
            "chains": int((is_5v5 & is_outside_origin).sum()),
        },
        {
            "cohort": "is_5v5_outside_origin_tracking_available",
            "chains": int((is_5v5 & is_outside_origin & has_tracking_available).sum()),
        },
        {
            "cohort": "is_5v5_outside_origin_tracking_available_no_error_flag",
            "chains": int(
                (
                    is_5v5
                    & is_outside_origin
                    & has_tracking_available
                    & has_no_tracking_error
                ).sum()
            ),
        },
    ]
)

cohort_waterfall["pct_of_all"] = cohort_waterfall["chains"] / len(chains_with_stints)

cohort_waterfall

,cohort,chains,pct_of_all
0,all_origin_shot_chains,48673,1.000000
1,valid_origin_time_and_stint,48671,0.999959
2,is_5v5,36357,0.746964
3,is_5v5_outside_origin,23754,0.488032
4,is_5v5_outside_origin_tracking_available,22336,0.458899
5,is_5v5_outside_origin_tracking_available_no_er...,22322,0.458612


## 8. Final Validation

Before saving, validate that the labeled dataset still has one row per chain and that all core labels are populated correctly.

The two known invalid origin links may remain missing origin time and stint match. Anything beyond that needs inspection before saving.

In [22]:
# Validate row preservation and strength-label completeness before saving

final_label_validation = {
    "rows": len(chains_with_stints),
    "unique_chain_ids": chains_with_stints["chain_id"].nunique(),
    "duplicate_chain_ids": chains_with_stints.duplicated("chain_id").sum(),
    "unknown_team_side": chains_with_stints["team_side"].eq("unknown").sum(),
    "missing_origin_time": chains_with_stints["anchor_origin_period_time"].isna().sum(),
    "missing_stint_match": chains_with_stints["game_stint"].isna().sum(),
    "origin_time_not_within_stint": (
        chains_with_stints["anchor_origin_period_time"].notna()
        & ~chains_with_stints["origin_time_within_stint"]
    ).sum(),
    "missing_is_5v5": chains_with_stints["is_5v5"].isna().sum(),
    "missing_team_n_skaters": chains_with_stints["team_n_skaters"].isna().sum(),
    "missing_opponent_n_skaters": chains_with_stints["opponent_n_skaters"].isna().sum(),
}

final_label_validation

{'rows': 48673,
 'unique_chain_ids': 48673,
 'duplicate_chain_ids': np.int64(0),
 'unknown_team_side': np.int64(0),
 'missing_origin_time': np.int64(2),
 'missing_stint_match': np.int64(2),
 'origin_time_not_within_stint': np.int64(0),
 'missing_is_5v5': np.int64(0),
 'missing_team_n_skaters': np.int64(2),
 'missing_opponent_n_skaters': np.int64(2)}

## 9. Save Labeled Dataset

Save the labeled chain dataset for the next notebook.

Output:

- `data/processed/origin_shot_sequences_labeled.parquet`

In [23]:
# Save the labeled origin-shot chain dataset for downstream analysis

output_path = DATA_PROCESSED / "origin_shot_sequences_labeled.parquet"

chains_with_stints.to_parquet(output_path, index=False)

print("Saved:", output_path)
print("Shape:", chains_with_stints.shape)

Saved: c:\Users\rinal\hockey-analytics\outside-shot-value\data\processed\origin_shot_sequences_labeled.parquet
Shape: (48673, 75)


In [24]:
# Confirm the saved labeled dataset can be read back successfully

saved_check = pd.read_parquet(DATA_PROCESSED / "origin_shot_sequences_labeled.parquet")

readback_validation = {
    "rows": len(saved_check),
    "unique_chain_ids": saved_check["chain_id"].nunique(),
    "duplicate_chain_ids": saved_check.duplicated("chain_id").sum(),
    "is_5v5_counts": saved_check["is_5v5"].value_counts(dropna=False).to_dict(),
}

readback_validation

{'rows': 48673,
 'unique_chain_ids': 48673,
 'duplicate_chain_ids': np.int64(0),
 'is_5v5_counts': {True: 36357, False: 12316}}